# Fit AGC to Illuminance: Processing Notebook

This notebook records the complete AGC-to-illuminance calibration workflow. It derives the shared camera lag, processes the selected recordings in memory, produces the linear-scale MATLAB input, and saves the fitted model under `derived/`.


## Setup

Load the two runnable derivation scripts. `deriveAGCLag.py` owns temporal alignment; `deriveEmpircalAGCAndIlluminance.py` owns point selection and MATLAB export.


In [ ]:
import importlib
import os
import sys
from pathlib import Path

from natsort import natsorted

PROJECT_ROOT = Path.cwd().parents[1]
DEFINE_DIR = PROJECT_ROOT / "code" / "defineWorldCameraCalibration"
DATA_PREP_DIR = DEFINE_DIR / "dataPrep"
for import_path in (DEFINE_DIR, DATA_PREP_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

import deriveAGCLag
import deriveEmpircalAGCAndIlluminance


## Select Recordings

Collect the raw `GKA` recording directories that will enter the processing fit. The current notebook preserves the existing exploratory scope of the run by processing the first four valid subject folders in the 2026 scripted indoor/outdoor dataset while skipping `FLIC_18`. Adjust `maximum_subjects_to_process` or the filtering rules here when intentionally regenerating the calibration data from a different recording set.


In [ ]:
flic_raw_path: str = "/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026"
maximum_subjects_to_process: int = 4
subjects_to_skip: set[str] = {"FLIC_18"}

recording_paths: list[str] = []
valid_subject_count: int = 0

for subject_dir in natsorted(os.listdir(flic_raw_path)):
    if valid_subject_count >= maximum_subjects_to_process:
        break

    if subject_dir in subjects_to_skip or subject_dir.startswith("."):
        continue

    subject_dir_path: str = os.path.join(flic_raw_path, subject_dir)
    if not os.path.isdir(subject_dir_path):
        continue

    for activity in natsorted(os.listdir(subject_dir_path)):
        if activity.startswith("."):
            continue

        activity_path: str = os.path.join(subject_dir_path, activity, "GKA")
        assert os.path.exists(activity_path), (
            f"Path does not exist: {activity_path}"
        )
        recording_paths.append(activity_path)

    valid_subject_count += 1

print(f"Selected {len(recording_paths)} recordings from {valid_subject_count} subjects.")


## Derive the Shared AGC Lag

This temporal-alignment stage converts minispect counts to illuminance, applies the empirical AGC kernel, chooses the shared lag that maximizes mean recording-level correlation, and writes `derived/cameraAGCLag.mat`.


In [ ]:
importlib.reload(deriveAGCLag)

lag_result = deriveAGCLag.derive_agc_lag(recording_paths)
lag_data_path = lag_result.output_path
lag_result


## Generate Final Linear-Scale MATLAB Input

Read the derived lag, process the same recordings in memory, apply the point-selection filters, and write `data/empircalAGCAndIlluminance.mat`. This `.mat` file is the formal input to the MATLAB piecewise log-log fit.


In [ ]:
importlib.reload(deriveEmpircalAGCAndIlluminance)

empirical_data_path = (
    deriveEmpircalAGCAndIlluminance.derive_empircal_agc_and_illuminance(
        recording_paths,
        lag_path=lag_data_path,
        maximum_saturation_percent=40.0,
        initial_samples_to_exclude=100,
    )
)
empirical_data_path


## Run MATLAB Piecewise Log-Log Fit

Run the MATLAB fitting script against `data/empircalAGCAndIlluminance.mat`. It reports and plots the piecewise log-log conversion and saves `derived/cameraAGCToIlluminanceFit.mat`, which Python loads during video processing.


In [ ]:
import matlab.engine

matlab_fit_dir = DEFINE_DIR / "utilities"
matlab_fit_script = matlab_fit_dir / "fitEmpircalAGCtoIlluminance.m"

matlab_engine = matlab.engine.start_matlab()
matlab_engine.cd(str(matlab_fit_dir), nargout=0)
matlab_engine.run(str(matlab_fit_script), nargout=0)
